In [1]:
import os
import sys

# Remove old Spark 3.5.9 configuration
os.environ.pop("SPARK_HOME", None)

# Tell PySpark to use the current Python 3.12
os.environ["PYSPARK_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

Python: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
SPARK_HOME: None
PYSPARK_PYTHON: c:\Users\bda\AppData\Local\Programs\Python\Python312\python.exe
PYSPARK_DRIVER_PYTHON: jupyter


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Fire Example") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark working")

c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark working


In [3]:
def create_sparkSession():
    spark = SparkSession.builder.appName("Fire Example").getOrCreate()
    return spark

In [4]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
filepath = r"D:\ABD-Lab (BDA)\data\sf-fire-calls.csv"

In [5]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')
    return df1

In [8]:
def clean_dataset(df):
    df1=df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2=df1.withColumn('Year',year(col('Date')))\
           .withColumn('Month',month(col('Date')))\
           .withColumn('Week',weekofyear(col('Date')))
    return df2

In [9]:
spark=create_sparkSession()
df=create_dataframe(spark,filepath)
df=clean_dataset(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [10]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [11]:
def mapSeason(data):
    if 2<data<6:
        return 'Spring'
    elif 5<data<9:
        return 'Summer'
    elif 8<data<12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF=udf(mapSeason, StringType())
clean_df=df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\bda\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [12]:
clean_df[clean_df['Season']=='Summer'].show()

+--------------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|            CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+--------------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|Citizen Assist / ...|  SF|  94124|Bayview Hunters P...|1.8833333|2002-06-01|2002|    6|  22|Summer|
|      Structure Fire|  SF|  94112|       Outer Mission|2.3166666|2002-06-01|2002|    6|  22|Summer|
|              Alarms|  SF|  94105|Financial Distric...|1.9833333|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94124|Bayview Hunters P...| 9.266666|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94122|     Sunset/Parkside|1.9166666|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94115|    Western Addition|3.9833333|2002-06-01|2002|    6|  22|Summer|
|    Medical Incident|  SF|  94109|           Japantown|      2.5|2002-06-01|2002|    6|  2